# LeetCode 3471 — Find the Largest Almost Missing Integer bằng Python

Link đề: https://leetcode.com/problems/find-the-largest-almost-missing-integer/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Sliding Window

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def largestInteger(self, nums: List[int], k: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu khái niệm **"almost missing"**: một số `x` chỉ được tính nếu nó xuất hiện trong **đúng một** cửa sổ (subarray liên tiếp) độ dài `k`, không phải đúng một **vị trí**.
2. Biết liệt kê các cửa sổ độ dài `k` bằng cắt lát `nums[i:i+k]` với `i` chạy từ `0` đến `n-k`, và dùng `set`, `Counter` để đếm.
3. Phân biệt "đếm số lần xuất hiện của giá trị" với "đếm số cửa sổ chứa giá trị" — đây là chỗ sai nhiều nhất của bài.
4. Viết 4 cách: duyệt từng giá trị + kiểm tra mọi cửa sổ, `Counter` trên `set` của từng cửa sổ, mảng hiệu (difference array) theo đoạn cửa sổ, và lối tắt `O(n)` chia theo `k == 1`, `k == n`, còn lại.
5. Tự viết test ngẫu nhiên có `random.seed` đối chiếu cả bốn cách và nhận diện các lỗi: bỏ sót cửa sổ cuối, trả `0` thay vì `-1`, đếm vị trí thay vì đếm cửa sổ.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên `k`. Một số `x` được gọi là **almost missing** nếu `x` xuất hiện trong **đúng một** subarray (dãy con liên tiếp) có độ dài `k` của `nums`.

Trả về số almost missing **lớn nhất**; nếu không có số nào như vậy thì trả về `-1`.

| Ví dụ | nums, k | Output | Giải thích |
|---|---|---|---|
| 1 | `[3,9,2,1,7]`, `k=3` | `7` | các cửa sổ: `[3,9,2]`, `[9,2,1]`, `[2,1,7]`; số `7` chỉ nằm trong cửa sổ cuối; `9` nằm trong hai cửa sổ nên không tính |
| 2 | `[3,9,7,2,1,7]`, `k=4` | `3` | cửa sổ: `[3,9,7,2]`, `[9,7,2,1]`, `[7,2,1,7]`; `3` chỉ nằm trong cửa sổ đầu; `7` nằm trong cả ba |
| 3 | `[0,0]`, `k=1` | `-1` | hai cửa sổ `[0]`, `[0]`: số `0` nằm trong hai cửa sổ, không có số nào xuất hiện trong đúng một |

Constraints: `1 <= nums.length <= 50`; `0 <= nums[i] <= 50`; `1 <= k <= nums.length`.

Chú ý: "xuất hiện trong đúng một subarray" đếm **số cửa sổ** chứa `x`, không phải số lần `x` xuất hiện trong `nums`. Ví dụ 3 có `0` hai lần nhưng kể cả nếu `nums=[0,0,0]`, `k=3` thì chỉ có một cửa sổ nên `0` vẫn là almost missing.

In [1]:
examples = [(([3, 9, 2, 1, 7], 3), 7), (([3, 9, 7, 2, 1, 7], 4), 3), (([0, 0], 1), -1)]
for (nums, k), expected in examples:
    n = len(nums)
    windows = [nums[i:i + k] for i in range(n - k + 1)]
    print(f"nums={nums}, k={k}: các cửa sổ = {windows}")
    for v in sorted(set(nums)):
        cnt = sum(1 for w in windows if v in w)
        print(f"   số {v} nằm trong {cnt} cửa sổ")
    print("   mong đợi:", expected)

nums=[3, 9, 2, 1, 7], k=3: các cửa sổ = [[3, 9, 2], [9, 2, 1], [2, 1, 7]]
   số 1 nằm trong 2 cửa sổ
   số 2 nằm trong 3 cửa sổ
   số 3 nằm trong 1 cửa sổ
   số 7 nằm trong 1 cửa sổ
   số 9 nằm trong 2 cửa sổ
   mong đợi: 7
nums=[3, 9, 7, 2, 1, 7], k=4: các cửa sổ = [[3, 9, 7, 2], [9, 7, 2, 1], [7, 2, 1, 7]]
   số 1 nằm trong 2 cửa sổ
   số 2 nằm trong 3 cửa sổ
   số 3 nằm trong 1 cửa sổ
   số 7 nằm trong 3 cửa sổ
   số 9 nằm trong 2 cửa sổ
   mong đợi: 3
nums=[0, 0], k=1: các cửa sổ = [[0], [0]]
   số 0 nằm trong 2 cửa sổ
   mong đợi: -1


## 3. Kiến thức nền

### 3.1. Cắt lát (slicing) và cửa sổ trượt

`nums[i:i+k]` lấy `k` phần tử bắt đầu từ vị trí `i` (nửa mở: không lấy phần tử `i+k`). Mảng độ dài `n` có đúng `n - k + 1` cửa sổ độ dài `k`, với `i = 0, 1, ..., n-k`. Vì vậy vòng lặp đúng là `range(n - k + 1)`; viết `range(n - k)` sẽ **bỏ sót cửa sổ cuối** (lỗi rất hay gặp).

In [2]:
nums = [3, 9, 2, 1, 7]
k = 3
n = len(nums)
print("số cửa sổ =", n - k + 1)
for i in range(n - k + 1):
    print(i, nums[i:i + k])

số cửa sổ = 3
0 [3, 9, 2]
1 [9, 2, 1]
2 [2, 1, 7]


### 3.2. Toán tử `in` và `set`

`v in w` kiểm tra `v` có nằm trong danh sách `w` hay không (duyệt tuyến tính). `set(w)` loại các phần tử trùng, nên khi chỉ quan tâm "có hay không" thì `set` là đủ: một cửa sổ có hai số `7` vẫn chỉ **tính một lần** cho giá trị `7`. Điều này quan trọng vì ta đếm **số cửa sổ chứa x**.

In [3]:
w = [7, 2, 1, 7]
print(7 in w, 5 in w)
print(set(w))            # {1, 2, 7} - hai số 7 gộp thành một

True False
{1, 2, 7}


### 3.3. `collections.Counter` / `dict` để đếm

`Counter` là một dạng `dict` chuyên đếm. `cnt[v] += 1` tăng số đếm của `v`; khóa chưa có tự có giá trị `0`. Ta dùng nó với nghĩa **"giá trị `v` nằm trong bao nhiêu cửa sổ"**.

In [4]:
from collections import Counter
cnt = Counter()
for w in [[3, 9, 2], [9, 2, 1], [2, 1, 7]]:
    for v in set(w):
        cnt[v] += 1
print(cnt)
print(cnt[100])         # khóa chưa có -> 0
print([v for v, c in cnt.items() if c == 1])

Counter({2: 3, 9: 2, 1: 2, 3: 1, 7: 1})
0
[3, 7]


### 3.4. `max` với `default` và giá trị trả về `-1`

`max(iterable, default=-1)` trả về phần tử lớn nhất, hoặc `-1` nếu dãy rỗng — đúng với yêu cầu "không có số almost missing thì trả `-1`". Nếu không có `default`, `max` của dãy rỗng gây `ValueError`.

In [5]:
print(max([3, 7, 5], default=-1))
print(max([], default=-1))
try:
    max([])
except ValueError as e:
    print("ValueError:", e)

7
-1
ValueError: max() iterable argument is empty


### 3.5. Mảng hiệu (difference array)

Muốn "cộng `1` cho mọi vị trí trong đoạn `[l, r]`" nhanh, ta làm `diff[l] += 1`, `diff[r+1] -= 1` rồi lấy tổng tích lũy. Cách 3 của notebook dùng nó để đếm số cửa sổ chứa mỗi giá trị mà không phải xét từng cửa sổ.

In [6]:
diff = [0] * 6
l, r = 1, 3            # cộng 1 cho vị trí 1..3
diff[l] += 1
diff[r + 1] -= 1
run, out = 0, []
for d in diff[:5]:
    run += d
    out.append(run)
print(out)             # [0, 1, 1, 1, 0]

[0, 1, 1, 1, 0]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Làm tay với `nums = [3,9,2,1,7]`, `k = 3`. Có `5 - 3 + 1 = 3` cửa sổ bắt đầu ở `0, 1, 2`:

```
vị trí:   0  1  2  3  4
nums  :   3  9  2  1  7
cửa sổ 0: [3  9  2]
cửa sổ 1:    [9  2  1]
cửa sổ 2:       [2  1  7]
```

Đếm "số cửa sổ chứa": `3` -> 1 (cửa sổ 0), `9` -> 2, `2` -> 3, `1` -> 2, `7` -> 1. Các giá trị có đếm bằng `1` là `3` và `7`; lớn nhất là `7`.

Nhận xét: phần tử ở **giữa** nằm trong nhiều cửa sổ; chỉ các phần tử gần **hai đầu mảng** mới nằm trong ít cửa sổ. Phần tử đầu `nums[0]` chỉ nằm trong cửa sổ `0`; phần tử cuối `nums[n-1]` chỉ nằm trong cửa sổ cuối.

## 5. Cách 1 — Brute Force / baseline

Với mỗi giá trị `v` (từ tập giá trị của `nums`), duyệt **tất cả** cửa sổ và đếm xem `v` có trong cửa sổ đó không. Nếu đếm đúng `1` thì `v` là ứng viên; lấy ứng viên lớn nhất.

Rất dễ đúng vì làm nguyên văn đề. Nút thắt: ứng với mỗi giá trị ta cắt `n-k+1` cửa sổ, mỗi cửa sổ dài `k` -> `O(n · n · k)`; còn dư thừa vì mọi cửa sổ đều được xét lại cho từng giá trị.

In [7]:
class SolutionBrute:
    def largestInteger(self, nums, k):
        n = len(nums)
        best = -1
        for v in set(nums):
            windows_with_v = 0
            for i in range(n - k + 1):
                if v in nums[i:i + k]:
                    windows_with_v += 1
            if windows_with_v == 1:
                best = max(best, v)
        return best

print(SolutionBrute().largestInteger([3, 9, 2, 1, 7], 3))
print(SolutionBrute().largestInteger([3, 9, 7, 2, 1, 7], 4))
print(SolutionBrute().largestInteger([0, 0], 1))

7
3
-1


**Độ phức tạp:** gọi `n = len(nums)`. Có tối đa `n` giá trị khác nhau × `n-k+1` cửa sổ × `O(k)` cho `in` -> `O(n² · k)` thời gian, `O(k)` bộ nhớ thêm cho lát cắt. Với `n <= 50` vẫn nhanh, nhưng ta có thể làm tốt hơn.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — đảo vòng lặp:** thay vì "mỗi giá trị duyệt mọi cửa sổ", ta "mỗi cửa sổ cập nhật các giá trị của nó" bằng một `Counter` -> mỗi cửa sổ chỉ xét một lần.

**Bước 2 — nhìn vị trí:** vị trí `j` nằm trong các cửa sổ bắt đầu tại `i` với `j-k+1 <= i <= j` và `0 <= i <= n-k`. Số cửa sổ chứa vị trí `j` là `min(j, n-k) - max(0, j-k+1) + 1`.

**Bước 3 — lối tắt (invariant theo trường hợp):**

- `k == n`: chỉ có **một** cửa sổ là chính `nums`, nên **mọi** giá trị đều almost missing -> đáp án `max(nums)`.
- `k == 1`: mỗi phần tử là một cửa sổ riêng; `x` almost missing khi nó xuất hiện **đúng một lần** trong `nums` -> lấy số lớn nhất có tần suất `1`.
- `1 < k < n`: mọi vị trí ở giữa (`1 <= j <= n-2`) nằm trong **ít nhất hai** cửa sổ, nên chỉ `nums[0]` và `nums[n-1]` có thể thỏa. Hơn nữa `x` phải xuất hiện **đúng một lần** trong cả mảng (nếu `x` xuất hiện thêm ở chỗ khác thì nó nằm thêm ít nhất một cửa sổ khác). Vậy đáp án là số lớn nhất trong `{nums[0], nums[n-1]}` mà tần suất toàn mảng bằng `1`.

In [8]:
nums, k = [3, 9, 2, 1, 7], 3
n = len(nums)
for j in range(n):
    lo, hi = max(0, j - k + 1), min(j, n - k)
    print(f"vị trí {j} (giá trị {nums[j]}): nằm trong {hi - lo + 1} cửa sổ, bắt đầu từ {lo} đến {hi}")

vị trí 0 (giá trị 3): nằm trong 1 cửa sổ, bắt đầu từ 0 đến 0
vị trí 1 (giá trị 9): nằm trong 2 cửa sổ, bắt đầu từ 0 đến 1
vị trí 2 (giá trị 2): nằm trong 3 cửa sổ, bắt đầu từ 0 đến 2
vị trí 3 (giá trị 1): nằm trong 2 cửa sổ, bắt đầu từ 1 đến 2
vị trí 4 (giá trị 7): nằm trong 1 cửa sổ, bắt đầu từ 2 đến 2


## 7. Các cách tiếp theo

Ta có 4 cách: (1) brute theo giá trị; (2) `Counter` trên `set` của từng cửa sổ; (3) mảng hiệu theo vị trí; (4) lối tắt `O(n)`.

### Cách 2 — `Counter` trên từng cửa sổ

Duyệt từng cửa sổ một lần; với mỗi giá trị **khác nhau** `v` trong cửa sổ, tăng `cnt[v]`. Cuối cùng lấy `max` của các `v` có `cnt[v] == 1`. Dùng `set(window)` để cửa sổ có hai số `7` chỉ góp một lần.

In [9]:
from collections import Counter

class SolutionCounter:
    def largestInteger(self, nums, k):
        n = len(nums)
        cnt = Counter()
        for i in range(n - k + 1):
            for v in set(nums[i:i + k]):
                cnt[v] += 1
        return max((v for v, c in cnt.items() if c == 1), default=-1)

print(SolutionCounter().largestInteger([3, 9, 2, 1, 7], 3))
print(SolutionCounter().largestInteger([3, 9, 7, 2, 1, 7], 4))
print(SolutionCounter().largestInteger([0, 0], 1))

7
3
-1


Độ phức tạp Cách 2: `n-k+1` cửa sổ × `O(k)` để tạo `set` -> `O(n·k)` thời gian, `O(n)` bộ nhớ cho `Counter` và `set` tạm.

### Cách 3 — Mảng hiệu theo khoảng cửa sổ của từng giá trị

Với mỗi giá trị `v`, tập cửa sổ chứa `v` là **hợp** của các đoạn `[max(0, j-k+1), min(j, n-k)]` qua mọi vị trí `j` có `nums[j] == v`. Ta đánh dấu từng đoạn lên mảng `covered` độ dài `n-k+1` (dùng mảng hiệu cho mỗi giá trị), rồi đếm số ô có tổng tích lũy `> 0`. Nếu đúng `1` ô được phủ thì `v` almost missing.

Cách này hữu ích khi ta muốn mở rộng sang `n` lớn: không phải cắt từng cửa sổ.

In [10]:
class SolutionDiff:
    def largestInteger(self, nums, k):
        n = len(nums)
        m = n - k + 1                      # số cửa sổ
        positions = {}
        for j, v in enumerate(nums):
            positions.setdefault(v, []).append(j)
        best = -1
        for v, plist in positions.items():
            diff = [0] * (m + 1)
            for j in plist:
                lo, hi = max(0, j - k + 1), min(j, n - k)
                diff[lo] += 1
                diff[hi + 1] -= 1
            run = covered = 0
            for t in range(m):
                run += diff[t]
                if run > 0:
                    covered += 1
            if covered == 1:
                best = max(best, v)
        return best

print(SolutionDiff().largestInteger([3, 9, 2, 1, 7], 3))
print(SolutionDiff().largestInteger([3, 9, 7, 2, 1, 7], 4))
print(SolutionDiff().largestInteger([0, 0], 1))

7
3
-1


Độ phức tạp Cách 3: mỗi vị trí `j` cập nhật `O(1)`, mỗi giá trị quét `m <= n` ô -> `O(n · d)` với `d` số giá trị khác nhau (`d <= n`), tức `O(n²)` trong trường hợp xấu nhất nhưng **không** phụ thuộc `k`. Bộ nhớ `O(n)`.

### Cách 4 — Lối tắt `O(n)` theo ba trường hợp

Dựa trên bước đột phá mục 6: dùng `Counter(nums)` để biết tần suất toàn mảng và chia ba trường hợp `k == n`, `k == 1`, `1 < k < n`.

In [11]:
class SolutionShortcut:
    def largestInteger(self, nums, k):
        n = len(nums)
        if k == n:
            return max(nums)
        freq = Counter(nums)
        if k == 1:
            return max((v for v, c in freq.items() if c == 1), default=-1)
        best = -1
        for v in (nums[0], nums[-1]):
            if freq[v] == 1:
                best = max(best, v)
        return best

print(SolutionShortcut().largestInteger([3, 9, 2, 1, 7], 3))
print(SolutionShortcut().largestInteger([3, 9, 7, 2, 1, 7], 4))
print(SolutionShortcut().largestInteger([0, 0], 1))

7
3
-1


Độ phức tạp Cách 4: `Counter` mất `O(n)`, phần còn lại `O(1)` hoặc `O(n)` -> **`O(n)` thời gian, `O(n)` bộ nhớ** (ở đây `n <= 50`, giá trị nhỏ nên thực tế là `O(1)` bộ nhớ nếu dùng mảng 51 ô).

## 8. TRACE từng bước

Trace Cách 2 với `nums = [3, 9, 7, 2, 1, 7]`, `k = 4` (`n = 6`, có `6-4+1 = 3` cửa sổ):

| cửa sổ `i` | `nums[i:i+4]` | `set` | `cnt` sau bước |
|---|---|---|---|
| 0 | `[3,9,7,2]` | `{3,9,7,2}` | `3:1, 9:1, 7:1, 2:1` |
| 1 | `[9,7,2,1]` | `{9,7,2,1}` | `3:1, 9:2, 7:2, 2:2, 1:1` |
| 2 | `[7,2,1,7]` | `{7,2,1}` | `3:1, 9:2, 7:3, 2:3, 1:2` |

Các `v` có `cnt == 1`: chỉ `3`. Đáp án `3`.

In [12]:
def trace_windows(nums, k):
    n = len(nums)
    cnt = Counter()
    for i in range(n - k + 1):
        w = nums[i:i + k]
        for v in set(w):
            cnt[v] += 1
        print(f"cửa sổ {i}: {w} -> set={sorted(set(w))} cnt={dict(sorted(cnt.items()))}")
    cand = sorted(v for v, c in cnt.items() if c == 1)
    print("ứng viên cnt == 1:", cand)
    ans = max(cand, default=-1)
    print("đáp án:", ans)
    return ans

trace_windows([3, 9, 7, 2, 1, 7], 4)
trace_windows([0, 0], 1)

cửa sổ 0: [3, 9, 7, 2] -> set=[2, 3, 7, 9] cnt={2: 1, 3: 1, 7: 1, 9: 1}
cửa sổ 1: [9, 7, 2, 1] -> set=[1, 2, 7, 9] cnt={1: 1, 2: 2, 3: 1, 7: 2, 9: 2}
cửa sổ 2: [7, 2, 1, 7] -> set=[1, 2, 7] cnt={1: 2, 2: 3, 3: 1, 7: 3, 9: 2}
ứng viên cnt == 1: [3]
đáp án: 3
cửa sổ 0: [0] -> set=[0] cnt={0: 1}
cửa sổ 1: [0] -> set=[0] cnt={0: 2}
ứng viên cnt == 1: []
đáp án: -1


-1

## 9. Code nộp LeetCode

In [13]:
from typing import List
from collections import Counter

class Solution:
    def largestInteger(self, nums: List[int], k: int) -> int:
        n = len(nums)
        if k == n:
            return max(nums)
        freq = Counter(nums)
        if k == 1:
            return max((v for v, c in freq.items() if c == 1), default=-1)
        best = -1
        for v in (nums[0], nums[-1]):
            if freq[v] == 1:
                best = max(best, v)
        return best

print(Solution().largestInteger([3, 9, 2, 1, 7], 3), Solution().largestInteger([3, 9, 7, 2, 1, 7], 4), Solution().largestInteger([0, 0], 1))

7 3 -1


## 10. Vì sao thuật toán đúng

- **Cách 1, 2:** làm đúng định nghĩa — đếm số cửa sổ chứa `v` rồi lấy các `v` có đếm bằng `1`. `set(window)` bảo đảm một cửa sổ chỉ tính một lần cho mỗi giá trị.
- **Cách 3:** cửa sổ bắt đầu tại `i` chứa vị trí `j` khi và chỉ khi `i <= j <= i+k-1`, tức `j-k+1 <= i <= j`, kết hợp `0 <= i <= n-k`. Hợp các đoạn này theo mọi vị trí của `v` chính là tập cửa sổ chứa `v`.
- **Cách 4:**
  - `k == n`: chỉ có cửa sổ `i = 0`, nó chứa mọi giá trị.
  - `k == 1`: cửa sổ `i` chỉ chứa `nums[i]`; `v` nằm trong số cửa sổ bằng số lần `v` xuất hiện.
  - `1 < k < n`: với vị trí `j` thỏa `1 <= j <= n-2`, các cửa sổ chứa `j` là `i` từ `lo = max(0, j-k+1)` đến `hi = min(j, n-k)`. Nếu `j <= k-1` thì `lo = 0` còn `hi >= 1` (vì `j >= 1` và `n-k >= 1`). Nếu `j >= k` thì `lo = j-k+1 < j` (do `k >= 2`) và `lo < n-k` (do `j < n-1`), nên `hi > lo`. Vậy mọi vị trí ở giữa nằm trong **ít nhất hai** cửa sổ và không bao giờ thỏa. Với `nums[0]` (chỉ cửa sổ `0`) và `nums[n-1]` (chỉ cửa sổ `n-k`) thì `v` thỏa khi không có vị trí nào khác chứa `v`: một vị trí khác ở giữa thêm cửa sổ mới, còn nếu `nums[0] == nums[n-1]` thì `v` nằm trong cả hai cửa sổ `0` và `n-k`, hai cửa sổ này khác nhau vì `k < n`.
- **Bộ test** đối chiếu cả bốn cách với nhau trên hàng nghìn input ngẫu nhiên nên đây là kiểm chứng thêm cho lập luận trên.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `d` là số giá trị khác nhau (`d <= n`).

| Cách | Time | Space |
|---|---|---|
| 1. Brute theo giá trị | `O(n² · k)` | `O(k)` |
| 2. `Counter` trên `set` từng cửa sổ | `O(n · k)` | `O(n)` |
| 3. Mảng hiệu theo vị trí | `O(n · d)` ≤ `O(n²)` | `O(n)` |
| 4. Lối tắt theo `k` | `O(n)` | `O(n)` |

Với `n <= 50`, `nums[i] <= 50` cả bốn cách đều tức thì. Cách 4 đẹp nhất về lý thuyết nhưng **dễ sai nếu không hiểu lý do**; cách 2 an toàn nhất khi làm bài thi.

## 12. Bộ test

Test 1: ví dụ đề và các edge case. Mọi input tuân thủ `1 <= n <= 50`, `0 <= nums[i] <= 50`, `1 <= k <= n`.

In [14]:
impls = [SolutionBrute(), SolutionCounter(), SolutionDiff(), SolutionShortcut(), Solution()]
tests = [
    (([3, 9, 2, 1, 7], 3), 7),
    (([3, 9, 7, 2, 1, 7], 4), 3),
    (([0, 0], 1), -1),
    (([5], 1), 5),                           # n = 1: một cửa sổ, một giá trị
    (([0], 1), 0),                           # giá trị 0 vẫn hợp lệ, khác -1
    (([1, 2, 3], 3), 3),                     # k = n -> max(nums)
    (([4, 4, 4], 3), 4),                     # k = n, mọi giá trị trùng
    (([1, 2, 2, 1], 1), -1),                 # k = 1, không giá trị nào xuất hiện đúng một lần
    (([1, 2, 2, 3], 1), 3),                  # k = 1, lấy max trong các giá trị xuất hiện một lần
    (([50, 0, 50], 2), -1),                  # 50 ở hai đầu: nằm trong hai cửa sổ khác nhau; 0 ở giữa: hai cửa sổ
    (([1, 1, 2, 3], 2), 3),                  # k < n: chỉ phần tử cuối 3 xuất hiện một lần
    (([7, 1, 2, 8], 3), 8),                  # hai đầu đều hợp lệ, lấy lớn nhất
    (([7, 1, 2, 7], 3), -1),                 # hai đầu trùng nhau -> mỗi giá trị nằm trong cả hai cửa sổ
    (([2, 9, 9, 2, 5], 2), 5),               # giá trị lớn 9 nằm ở giữa
]
for (nums, k), exp in tests:
    assert 1 <= len(nums) <= 50 and all(0 <= v <= 50 for v in nums) and 1 <= k <= len(nums)
    for im in impls:
        got = im.largestInteger(list(nums), k)
        assert got == exp, (type(im).__name__, nums, k, got, exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 14


Test 2: ngẫu nhiên có seed. Dùng nhiều dải giá trị (`0..3` để có nhiều trùng lặp, `0..50` để có nhiều giá trị khác nhau) và mọi `k` hợp lệ. Oracle độc lập: mô phỏng hoàn toàn theo định nghĩa bằng chuỗi các cửa sổ và `sum`.

In [15]:
import random
random.seed(3471)

def oracle(nums, k):
    n = len(nums)
    windows = [nums[i:i + k] for i in range(n - k + 1)]
    best = -1
    for x in range(0, 51):
        if sum(1 for w in windows if x in w) == 1:
            best = x
    return best

for t in range(3000):
    n = random.randint(1, 50)
    hi_val = random.choice([1, 3, 10, 50])
    nums = [random.randint(0, hi_val) for _ in range(n)]
    k = random.randint(1, n)
    assert 1 <= n <= 50 and all(0 <= v <= 50 for v in nums) and 1 <= k <= n
    ref = oracle(nums, k)
    for im in impls:
        assert im.largestInteger(list(nums), k) == ref, (type(im).__name__, nums, k)
print("3000 test ngẫu nhiên OK")

3000 test ngẫu nhiên OK


Test 3: kiểm chứng trực tiếp tuyên bố ở mục 10 — với `1 < k < n`, đáp án (nếu khác `-1`) luôn là `nums[0]` hoặc `nums[-1]`; với mọi `k`, không input nào cho kết quả `x` mà `x` không nằm trong `nums`. Thêm: không sửa input.

In [16]:
random.seed(71)
for _ in range(2000):
    n = random.randint(3, 50)
    nums = [random.randint(0, 20) for _ in range(n)]
    k = random.randint(2, n - 1)                  # 1 < k < n
    snapshot = list(nums)
    ans = Solution().largestInteger(nums, k)
    assert nums == snapshot
    assert ans == -1 or ans in (nums[0], nums[-1])
    assert ans == -1 or ans in nums
print("tính chất hai đầu / không sửa input: OK")

tính chất hai đầu / không sửa input: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: đếm số lần xuất hiện trong `nums` thay vì số cửa sổ

In [17]:
def wrong_count_positions(nums, k):
    freq = Counter(nums)                       # SAI: đếm vị trí, không đếm cửa sổ
    return max((v for v, c in freq.items() if c == 1), default=-1)

nums, k = [3, 9, 2, 1, 7], 3
print("sai :", wrong_count_positions(nums, k))
print("đúng:", Solution().largestInteger(nums, k))
assert wrong_count_positions(nums, k) != Solution().largestInteger(nums, k)

sai : 9
đúng: 7


Vì sao sai: `9` xuất hiện **một lần** trong `nums` nhưng nằm trong **hai** cửa sổ nên không thỏa; code sai trả `9` thay vì `7`. **Sửa:** đếm số cửa sổ chứa giá trị (Cách 1–3), hoặc dùng lối tắt chỉ cho `k == 1`.

### Lỗi 2: `range(n - k)` bỏ sót cửa sổ cuối

In [18]:
def wrong_missing_last_window(nums, k):
    n = len(nums)
    cnt = Counter()
    for i in range(n - k):                     # SAI: phải là n - k + 1
        for v in set(nums[i:i + k]):
            cnt[v] += 1
    return max((v for v, c in cnt.items() if c == 1), default=-1)

nums, k = [3, 9, 2, 1, 7], 3
print("sai :", wrong_missing_last_window(nums, k))
print("đúng:", Solution().largestInteger(nums, k))
assert wrong_missing_last_window(nums, k) != Solution().largestInteger(nums, k)

sai : 3
đúng: 7


Vì sao sai: cửa sổ cuối `[2,1,7]` bị bỏ nên `7` không bao giờ được đếm; code sai trả `3`. **Sửa:** có `n - k + 1` cửa sổ, nên `range(n - k + 1)`.

### Lỗi 3: trả `0` thay vì `-1` khi không có đáp án

In [19]:
def wrong_default_zero(nums, k):
    n = len(nums)
    cnt = Counter()
    for i in range(n - k + 1):
        for v in set(nums[i:i + k]):
            cnt[v] += 1
    best = 0                                   # SAI: 0 là một giá trị hợp lệ của nums
    for v, c in cnt.items():
        if c == 1:
            best = max(best, v)
    return best

nums, k = [0, 0], 1
print("sai :", wrong_default_zero(nums, k))
print("đúng:", Solution().largestInteger(nums, k))
assert wrong_default_zero(nums, k) != Solution().largestInteger(nums, k)

sai : 0
đúng: -1


Vì sao sai: `0` cũng có thể là một đáp án thật (ví dụ `nums=[0], k=1` thì đáp án là `0`), nên không dùng `0` làm "chưa có". **Sửa:** khởi tạo `best = -1` hoặc `max(..., default=-1)`.

### Lỗi 4: dùng `list` thay vì `set` trong một cửa sổ

In [20]:
def wrong_no_set(nums, k):
    n = len(nums)
    cnt = Counter()
    for i in range(n - k + 1):
        for v in nums[i:i + k]:                # SAI: số trùng trong cùng cửa sổ bị đếm nhiều lần
            cnt[v] += 1
    return max((v for v, c in cnt.items() if c == 1), default=-1)

nums, k = [5, 5, 1], 3
print("sai :", wrong_no_set(nums, k))
print("đúng:", Solution().largestInteger(nums, k))
assert wrong_no_set(nums, k) != Solution().largestInteger(nums, k)

sai : 1
đúng: 5


Vì sao sai: chỉ có một cửa sổ `[5,5,1]`, nên cả `5` và `1` đều almost missing và đáp án là `5`; nhưng nếu đếm theo từng phần tử thì `5` bị đếm hai lần, loại nhầm. **Sửa:** `set(window)` hoặc kiểm tra `v in window` một lần cho mỗi cửa sổ.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Cụm "xuất hiện trong **đúng một** subarray độ dài `k`" -> nghĩ **sliding window + đếm số cửa sổ chứa**.
- Với `n <= 50` cứ làm brute/`Counter` cho chắc; lối tắt `O(n)` chỉ là bonus để hiểu cấu trúc.
- Nhớ số cửa sổ là `n - k + 1`; đừng quên cửa sổ cuối.
- Tách ba trường hợp biên: `k == 1`, `k == n`, còn lại.
- Phần tử **gần hai đầu** thuộc ít cửa sổ nhất — mẫu hình quen thuộc của bài sliding window.
- Luôn dùng `-1` cho "không có", không dùng `0` khi `0` là giá trị hợp lệ.

## 15. Biến thể của bài

1. Trả về số almost missing **nhỏ nhất**, hoặc **số lượng** số almost missing.
2. Số `x` phải xuất hiện trong đúng `m` cửa sổ (tổng quát `m` thay cho `1`): Cách 2 và 3 áp dụng ngay.
3. Giá trị lớn (`nums[i]` đến `10^9`): dùng `dict` thay vì mảng đếm.
4. `n` rất lớn (`10^5`): bỏ Cách 1; dùng Cách 3 (mảng hiệu) hoặc lối tắt.
5. Subarray không liên tiếp (subsequence) -> bài khác hẳn, đếm tổ hợp.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute theo giá trị | mỗi giá trị duyệt mọi cửa sổ | `O(n² · k)` | `O(k)` | oracle, `n` rất nhỏ |
| 2. `Counter` trên `set` | mỗi cửa sổ cập nhật một lần | `O(n · k)` | `O(n)` | **an toàn, dễ viết** |
| 3. Mảng hiệu | đoạn cửa sổ chứa mỗi vị trí | `O(n · d)` | `O(n)` | `k` lớn, học kỹ thuật |
| 4. Lối tắt theo `k` | chỉ xét hai đầu mảng | `O(n)` | `O(n)` | tối ưu, cần hiểu bằng chứng |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `largestInteger([1, 2, 3, 4], 2)` bằng bao nhiêu?

**Bài 2 (tính tay):** `nums = [4, 6, 4, 8]`, `k = 3`. Liệt kê các cửa sổ và cho biết mỗi số nằm trong mấy cửa sổ; đáp án là gì?

**Bài 3 (điền TODO):** hoàn thành `windows_containing(nums, k, v)` đếm số cửa sổ độ dài `k` chứa `v`.

**Bài 4 (tự viết lại):** viết `count_almost_missing(nums, k)` trả về **số lượng** các giá trị almost missing.

**Bài 5 (mini challenge):** viết `smallest_almost_missing(nums, k)` trả về số almost missing **nhỏ nhất** (hoặc `-1`).

In [21]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().largestInteger([1, 2, 3, 4], 2))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def windows_containing(nums, k, v):
    # TODO: duyệt i từ 0 đến n - k, đếm cửa sổ nums[i:i+k] có chứa v
    return 0

4


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** liệt kê cửa sổ rồi đếm; nhớ có `n - k + 1` cửa sổ.

**Mức 2 (vừa):** Bài 3: `for i in range(len(nums) - k + 1): if v in nums[i:i+k]: c += 1`. Bài 4: dùng `Counter` các cửa sổ như Cách 2 rồi đếm các `v` có `c == 1`. Bài 5: giống Cách 2 nhưng dùng `min(..., default=-1)`.

**Mức 3 (gần đáp án):** Bài 1: các cửa sổ `[1,2]`, `[2,3]`, `[3,4]`; `1` và `4` nằm trong một cửa sổ, `2` và `3` nằm trong hai -> đáp án `4`. Bài 2: cửa sổ `[4,6,4]`, `[6,4,8]`; `4` nằm trong hai cửa sổ, `6` hai cửa sổ, `8` một cửa sổ -> đáp án `8`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1
assert Solution().largestInteger([1, 2, 3, 4], 2) == 4

# Bài 2: cửa sổ [4,6,4], [6,4,8]; 8 chỉ ở cửa sổ cuối
assert Solution().largestInteger([4, 6, 4, 8], 3) == 8

# Bài 3
def windows_containing(nums, k, v):
    c = 0
    for i in range(len(nums) - k + 1):
        if v in nums[i:i + k]:
            c += 1
    return c
assert windows_containing([3, 9, 2, 1, 7], 3, 9) == 2
assert windows_containing([3, 9, 2, 1, 7], 3, 7) == 1
assert windows_containing([3, 9, 2, 1, 7], 3, 100) == 0

# Bài 4
def count_almost_missing(nums, k):
    cnt = Counter()
    for i in range(len(nums) - k + 1):
        for v in set(nums[i:i + k]):
            cnt[v] += 1
    return sum(1 for c in cnt.values() if c == 1)
assert count_almost_missing([3, 9, 2, 1, 7], 3) == 2       # 3 và 7
assert count_almost_missing([0, 0], 1) == 0
assert count_almost_missing([1, 2, 3], 3) == 3

# Bài 5
def smallest_almost_missing(nums, k):
    cnt = Counter()
    for i in range(len(nums) - k + 1):
        for v in set(nums[i:i + k]):
            cnt[v] += 1
    return min((v for v, c in cnt.items() if c == 1), default=-1)
assert smallest_almost_missing([3, 9, 2, 1, 7], 3) == 3
assert smallest_almost_missing([0, 0], 1) == -1
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đếm số cửa sổ chứa mỗi giá trị":**

```python
cnt = Counter()
for i in range(len(nums) - k + 1):
    for v in set(nums[i:i + k]):
        cnt[v] += 1
return max((v for v, c in cnt.items() if c == 1), default=-1)
```

**Checklist trước khi submit:**

- [ ] Đếm **số cửa sổ** chứa giá trị, không phải số lần xuất hiện?
- [ ] Số cửa sổ là `n - k + 1` (có cửa sổ cuối)?
- [ ] Trong một cửa sổ dùng `set` để mỗi giá trị chỉ tính một lần?
- [ ] Không có đáp án thì trả `-1` (không phải `0`)?
- [ ] Đã thử `k == 1`, `k == n`, `n == 1`, mảng toàn một giá trị?
- [ ] Chữ ký đúng `largestInteger(self, nums: List[int], k: int) -> int`?